# Lesson 07 - Planleggingsdesignmønster

Denne notatboken demonstrerer **Planleggingsdesignmønsteret** for AI-agenter ved bruk av Microsoft Agent Framework.
Du vil lære hvordan du kan dele opp komplekse reisespørsmål i strukturerte deloppgaver, tildele dem til spesialistagenter,
og utføre den resulterende planen — alt ved bruk av strukturert utdata drevet av Pydantic-modeller.


## Oppsett


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Oppgavedekomponering

Oppgavedekomponering er kjernen i planleggingsdesignmønsteret. I stedet for å be en enkelt agent om å håndtere en kompleks forespørsel fra start til slutt, deler vi problemet opp i mindre, veldefinerte **deloppgaver**. Hver deloppgave tilordnes en spesialistagent (f.eks. flyreiser, hoteller, aktiviteter) med klare prioriteringer og avhengighetsrekkefølge.

Denne tilnærmingen gir flere fordeler:
- **Klarhet**: hver deloppgave har ett enkelt ansvar.
- **Parallellitet**: uavhengige deloppgaver kan kjøre samtidig.
- **Pålitelighet**: feil isoleres til individuelle deloppgaver.
- **Budsjettsporing**: kostnader estimeres per deloppgave og summeres opp.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Lage en planleggingsagent med strukturert utdata

Planleggingsagenten fungerer som en **resepsjonskoordinator**. Gitt en overordnet reiseforespørsel produserer den en strukturert `TravelPlan` — som bryter ned forespørselen i deloppgaver, setter prioriteringer, og identifiserer avhengigheter slik at en concierge eller utførelseslag kan utføre arbeidet.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Utføre en plan med spesialistverktøy

Når resepsjonisten har utarbeidet en strukturert plan, utfører **concierge-agenten** den.
Hvert spesialistverktøy håndterer én kategori av deloppgaver (fly, hotell, aktiviteter). Concierge-agenten
går gjennom deloppgavene i planen i avhengighetsrekkefølge og sender hver enkelt til
det passende verktøyet.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Sammendrag

I denne leksjonen lærte du **Planleggingsdesignmønsteret** for AI-agenter:

1. **Oppgavedekomponering** — En planleggingsagent i resepsjonen bryter ned en kompleks reiseforespørsel i strukturerte deloppgaver ved hjelp av Pydantic-modeller, og tildeler hver til en spesialistagent med prioriteringer og avhengigheter.
2. **Strukturert Utdata** — Ved å sende en `response_format` returnerer agenten et validert `TravelPlan`-objekt i stedet for fritekst, noe som gjør videre prosessering pålitelig.
3. **Planutførelse** — En concierge-agent går gjennom deloppgavene ved hjelp av spesialisert verktøy (`book_flight`, `reserve_hotel`, `book_activity`) for å utføre planen og rapportere resultater.

Dette mønsteret skiller *hva som skal gjøres* (planlegging) fra *hvordan det skal gjøres* (utførelse), noe som gjør agenter mer modulære, testbare og enklere å utvide.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi søker nøyaktighet, vennligst vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på sitt originale språk bør anses som den autoritative kilden. For kritisk informasjon anbefales profesjonell, menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
